# LaunchReady: benchmark dry run on Kaggle T4 x2 (Qwen3-4B)

**Why:** prove `bench/run_session_5090.sh` works end to end on a free GPU before renting an
RTX 5090 (BUILD_PLAN Phase 5 step 5). Same flow as the 5090 session, with Qwen/Qwen3-4B:

1. a startup-only "card" server: Qwen3-4B's card line (expected to **fail** on vLLM 0.32.0, which removed `--enable-reasoning`), then the same line without that flag
2. a vLLM benchmark server that tries FP8 KV first and falls back without it
3. speculative decoding on/off (ngram stands in for NVIDIA's DSpark drafter)
4. `llama-server -np 8` built from llama.cpp v0.8.0, and Ollama at concurrency 1
5. a small GSM8K/IFEval subset through lm-eval, with AIPerf for every benchmark point

Numbers from this notebook are a **dry run of the scripts**, not published results.

**To run:** Kaggle → Settings → Accelerator **GPU T4 x2**, Internet **on** → *Run all*
(~1.5–2.5 h: llama.cpp CUDA build ~15 min, three vLLM startups, Ollama pull).
Then download `/kaggle/working/launchready_dryrun.zip` from the Output panel and share it.

Before running, push the commit named in `LAUNCHREADY_REF` to GitHub.

In [ ]:
import os
import subprocess
import sys

# Pinned versions: change them here and nowhere else.
LAUNCHREADY_REF = "phase5-bench"  # branch or commit pushed to GitHub
PINS = ["torch==2.13.0", "vllm==0.32.0", "uv==0.13.6"]  # vLLM 0.32.0 worked on T4 in Phase 3
REPO_DIR = "/kaggle/working/launchready"

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout)
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    raise RuntimeError("No GPU: Settings → Accelerator → GPU T4 x2, then Run all.") from e

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/Tanmaypathak18/launchready", REPO_DIR], check=True
    )
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", LAUNCHREADY_REF], check=True)
print(
    subprocess.run(
        ["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True
    ).stdout
)

print("Installing pinned packages (5–10 min)...")
pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", *PINS],
    capture_output=True,
    text=True,
)
if pip.returncode:
    print(pip.stdout[-4000:], pip.stderr[-4000:])
    raise RuntimeError("pip install failed (see above). Is Internet on?")
print(subprocess.run(["vllm", "--version"], capture_output=True, text=True).stdout)

## Run the session

This runs exactly what the 5090 session runs, with the Kaggle matrix. `HOURLY_PRICE=0`
(Kaggle is free), so $ per 1M tokens comes out as 0. `MAX_HOURS=3` is the hard stop.
Output streams below; the full logs are under `results_dryrun/bench/<session>/logs/`.

In [ ]:
env = {
    **os.environ,
    "MATRIX": "bench/matrices/kaggle_qwen3_4b.yaml",
    "HOURLY_PRICE": "0",
    "MAX_HOURS": "3",
    "BENCH_PROVIDER": "kaggle",
    "BENCH_CONTAINER_IMAGE": "kaggle notebook (pip vllm==0.32.0)",
}
if "HF_TOKEN" not in env:
    try:
        from kaggle_secrets import UserSecretsClient

        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret (optional: none of these models is gated).")

proc = subprocess.Popen(
    ["bash", "bench/run_session_5090.sh"],
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    print(line, end="")
print("exit code", proc.wait())

## Check the outputs

Every startup attempt, every run and every quality task should have its own JSON. The
`card` variant is expected to fail; everything else should pass.

In [ ]:
import glob
import json

import pandas as pd

os.chdir(REPO_DIR)
[session_dir] = sorted(glob.glob("results_dryrun/bench/dryrun_kaggle_qwen3_4b_*"))[-1:]
print("session:", session_dir)

startup = [json.load(open(p)) for p in sorted(glob.glob(f"{session_dir}/startup/*.json"))]
display(
    pd.DataFrame(
        [
            {
                "server": s["server"],
                "variant": s["variant"],
                "passed": s["passed"],
                "health": s["health"],
                "s_to_healthy": s["seconds_to_healthy"],
                "engine": s.get("engine_version"),
                "vllm": s.get("vllm_capacity"),
            }
            for s in startup
        ]
    )
)

runs = pd.read_csv("results_dryrun/summary.csv")
cols = [
    "run_id",
    "status",
    "ttft_ms_p50",
    "ttft_ms_p99",
    "itl_ms_p50",
    "itl_ms_p99",
    "output_tps_total",
    "output_tps_per_user_p50",
    "osl_mean",
    "gpu_mem_used_mib_peak",
    "gpu_power_w_mean",
    "note",
]
display(runs[cols])

if os.path.exists("results_dryrun/quality_summary.csv"):
    display(
        pd.read_csv("results_dryrun/quality_summary.csv")[
            ["server", "task", "metric", "value", "thinking", "gguf_tokenizer_pre", "status"]
        ]
    )
print(open(f"{session_dir}/session_stages.tsv").read())

problems = [r for r in runs.to_dict("records") if r["status"] != "ok"]
print(f"{len(runs)} runs, {len(problems)} not ok", [p["run_id"] for p in problems])

In [ ]:
import shutil

# All dry-run results, for download from the Output panel.
zip_path = shutil.make_archive(
    "/kaggle/working/launchready_dryrun", "zip", REPO_DIR, "results_dryrun"
)
print(zip_path, os.path.getsize(zip_path) // 2**20, "MiB")